# 資料前處理(Label encoding、 One hot encoding)
這兩個編碼方式的目的是為了將類別 (categorical)或是文字(text)的資料轉換成數字，而讓程式能夠更好的去理解及運算。
> Label encoding : 把每個類別 mapping 到某個整數，不會增加新欄位

> One hot encoding : 為每個類別新增一個欄位，用 0/1 表示是否

![](images/Encoder.PNG)


## Encoding Categorical features (or label)
![](images/Encoding.PNG)


In [1]:
import pandas as pd
import numpy as np


In [2]:
df = pd.DataFrame({'blood':['A','B','AB','O','B'], 
                   'Y':['high','low','high','mid','mid'],
                   'Z':[np.nan,np.nan,-1196,72,83]});
df

,blood,Y,Z
0,A,high,NaN
1,B,low,NaN
2,AB,high,-1196.0
3,O,mid,72.0
4,B,mid,83.0


# 方法一：sklearn - label encoder + onehot encoder
>onehot encoder要用2D array，若維度所以要用reshape(-1,1)<br>
>onehot encoder要數字，若資料文文字要先用label encoder轉數字

In [3]:
from sklearn.preprocessing import LabelEncoder

labelencoder = LabelEncoder()
df_le = df.copy()
df_le['blood_label'] = labelencoder.fit_transform(df_le['blood'])
print(dict(zip(labelencoder.classes_, labelencoder.transform(labelencoder.classes_))))
df_le


{'A': np.int64(0), 'AB': np.int64(1), 'B': np.int64(2), 'O': np.int64(3)}


,blood,Y,Z,blood_label
0,A,high,NaN,0
1,B,low,NaN,2
2,AB,high,-1196.0,1
3,O,mid,72.0,3
4,B,mid,83.0,2


In [4]:
from sklearn.preprocessing import OneHotEncoder

blood_label = labelencoder.fit_transform(df['blood']).reshape(-1, 1)
onehotencoder = OneHotEncoder(sparse_output=False, dtype=int)
blood_ohe = onehotencoder.fit_transform(blood_label)
blood_ohe_df = pd.DataFrame(blood_ohe, columns=[f'blood_{name}' for name in labelencoder.classes_])
pd.concat([df, blood_ohe_df], axis=1)


,blood,Y,Z,blood_A,blood_AB,blood_B,blood_O
0,A,high,NaN,1,0,0,0
1,B,low,NaN,0,0,1,0
2,AB,high,-1196.0,0,1,0,0
3,O,mid,72.0,0,0,0,1
4,B,mid,83.0,0,0,1,0


## One hot encoding
One Hot encoding的編碼邏輯為將類別拆成多個行(column)，每個列中的數值由1、0替代，當某一列的資料存在的該行的類別則顯示1，反則顯示0。

然在指定column進行編碼的情形下，One hot encoding<b>無法直接對字串進行編碼，必須先透過Label encoding將字串以數字取代後再進行One hot encoding處理。</b>

> categorical_features = [0]: 表示欲在data上執行One hot encoding的index為0

> data_le: 為經過Label encoding編碼的資料(註:OneHotEncoder的輸入要為2-D array，而Label encoding為1-D array)


OneHotEncoder會轉出scipy.csr_matrix資料結構用.toarray()轉array
從結果可以知道，數字0的column 代表的是A、數字1的column 代表的是B，而數字2的column 代表的是AB。
除了轉換字串外，One hot encoding也可以轉換數字。在此處的data就不需要先經過Label encoding編碼

```python
# importing one hot encoder from sklearn 
# There are changes in OneHotEncoder class 
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer 
   
# creating one hot encoder object with categorical feature 0 
# indicating the first column 
columnTransformer = ColumnTransformer([('encoder', 
                                        OneHotEncoder(), 
                                        [0])], 
                                      remainder='passthrough') 
  
data = np.array(columnTransformer.fit_transform(data), dtype = str) 
```

In [5]:
# importing one hot encoder from sklearn 
# There are changes in OneHotEncoder class 
from sklearn.preprocessing import OneHotEncoder 
from sklearn.compose import ColumnTransformer 

# creating one hot encoder object with categorical feature 0 
# indicating the first column 
columnTransformer = ColumnTransformer([('encoder', 
                                        OneHotEncoder(sparse_output=False, dtype=int), 
                                        [0])], 
                                      remainder='passthrough') 

encoded_data = columnTransformer.fit_transform(df)
encoded_columns = columnTransformer.get_feature_names_out()
pd.DataFrame(encoded_data, columns=encoded_columns)


,encoder__blood_A,encoder__blood_AB,encoder__blood_B,encoder__blood_O,remainder__Y,remainder__Z
0,1,0,0,0,high,NaN
1,0,0,1,0,low,NaN
2,0,1,0,0,high,-1196.0
3,0,0,0,1,mid,72.0
4,0,0,1,0,mid,83.0


# 方法二：Keras - label encoder + to_categorical
>to_categorical要數字，若資料文文字要先用label encoder轉數字

In [6]:
from sklearn.preprocessing import LabelEncoder

# Keras-compatible one-hot helper?? NumPy ??????????????????
def to_categorical(labels, num_classes=None):
    labels = np.asarray(labels, dtype=int).ravel()
    if num_classes is None:
        num_classes = labels.max() + 1
    result = np.zeros((labels.size, num_classes), dtype=int)
    result[np.arange(labels.size), labels] = 1
    return result

class np_utils:
    to_categorical = staticmethod(to_categorical)

df = pd.DataFrame({'blood':['A','B','AB','O','B'], 
                   'Y':['high','low','high','mid','mid'],
                   'Z':[np.nan,np.nan,-1196,72,83]})

# label encoder 
labelencoder = LabelEncoder()
encoded_Y = labelencoder.fit_transform(df['blood'])
print(dict(zip(labelencoder.classes_, labelencoder.transform(labelencoder.classes_))))
print(encoded_Y)

# convert integers to one hot encoding
dummy_y = np_utils.to_categorical(encoded_Y)
pd.DataFrame(dummy_y, columns=[f'blood_{name}' for name in labelencoder.classes_])


{'A': np.int64(0), 'AB': np.int64(1), 'B': np.int64(2), 'O': np.int64(3)}
[0 2 1 3 2]


,blood_A,blood_AB,blood_B,blood_O
0,1,0,0,0
1,0,0,1,0
2,0,1,0,0
3,0,0,0,1
4,0,0,1,0


## 方法三：pd.get_dummies方法
![](images/Encoding_pd.PNG)
pd.get_dummies(df)
>get_dummies可以直接轉字串，反而無法轉換數字<br>
>get_dummies沒指定columns，會全部轉換

In [7]:
df = pd.DataFrame({'blood':['A','B','AB','O','B'], 
                   'Y':['high','low','high','mid','mid'],
                   'Z':[np.nan,np.nan,-1196,72,83]})
pd.get_dummies(df, dtype=int)


,Z,blood_A,blood_AB,blood_B,blood_O,Y_high,Y_low,Y_mid
0,NaN,1,0,0,0,1,0,0
1,NaN,0,0,1,0,0,1,0
2,-1196.0,0,1,0,0,1,0,0
3,72.0,0,0,0,1,0,0,1
4,83.0,0,0,1,0,0,0,1


## 練習一：sklearn - label encoder + onehot encoder
下面的資料可以看到country那欄皆為字串， 大部分的模型都是基於數學運算，字串無法套入數學模型進行運算，<br>
在此先對其進行Label encoding編碼，我們從 sklearn library中導入 LabelEncoder class，對第一行資料進行fit及transform並取代之。

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

country = ['Taiwan', 'Australia', 'Ireland', 'Australia', 'Ireland', 'Taiwan']
age = [25, 30, 45, 35, 22, 36]
salary = [20000, 32000, 59000, 60000, 43000, 52000]
dic = {'Country': country, 'Age': age, 'Salary': salary}
data = pd.DataFrame(dic)
display(data)

labelencoder = LabelEncoder()
data_le = data.copy()
data_le['Country'] = labelencoder.fit_transform(data_le['Country'])
display(data_le)

onehotencoder = OneHotEncoder(sparse_output=False, dtype=int)
country_ohe = onehotencoder.fit_transform(data_le[['Country']])
country_ohe_df = pd.DataFrame(country_ohe, columns=[f'Country_{name}' for name in labelencoder.classes_])
pd.concat([country_ohe_df, data[['Age', 'Salary']]], axis=1)


,Country,Age,Salary
0,Taiwan,25,20000
1,Australia,30,32000
2,Ireland,45,59000
3,Australia,35,60000
4,Ireland,22,43000
5,Taiwan,36,52000


,Country,Age,Salary
0,2,25,20000
1,0,30,32000
2,1,45,59000
3,0,35,60000
4,1,22,43000
5,2,36,52000


,Country_Australia,Country_Ireland,Country_Taiwan,Age,Salary
0,0,0,1,25,20000
1,1,0,0,30,32000
2,0,1,0,45,59000
3,1,0,0,35,60000
4,0,1,0,22,43000
5,0,0,1,36,52000


## 練習二：Keras - label encoder + to_categorical

In [9]:
from sklearn.preprocessing import LabelEncoder

country = ['Taiwan', 'Australia', 'Ireland', 'Australia', 'Ireland', 'Taiwan']
age = [25, 30, 45, 35, 22, 36]
salary = [20000, 32000, 59000, 60000, 43000, 52000]
dic = {'Country': country, 'Age': age, 'Salary': salary}
data = pd.DataFrame(dic)
display(data)

labelencoder = LabelEncoder()
encoded_country = labelencoder.fit_transform(data['Country'])
country_categorical = np_utils.to_categorical(encoded_country)
country_categorical_df = pd.DataFrame(country_categorical, columns=[f'Country_{name}' for name in labelencoder.classes_])
pd.concat([country_categorical_df, data[['Age', 'Salary']]], axis=1)


,Country,Age,Salary
0,Taiwan,25,20000
1,Australia,30,32000
2,Ireland,45,59000
3,Australia,35,60000
4,Ireland,22,43000
5,Taiwan,36,52000


,Country_Australia,Country_Ireland,Country_Taiwan,Age,Salary
0,0,0,1,25,20000
1,1,0,0,30,32000
2,0,1,0,45,59000
3,1,0,0,35,60000
4,0,1,0,22,43000
5,0,0,1,36,52000


## 練習三：Pandas.get_dummies
>　get_dummies : 僅能將字串轉換為One hot encoding表示形式， 沒指定columns會全部轉換。

In [10]:
country = ['Taiwan', 'Australia', 'Ireland', 'Australia', 'Ireland', 'Taiwan']
age = [25, 30, 45, 35, 22, 36]
salary = [20000, 32000, 59000, 60000, 43000, 52000]
dic = {'Country': country, 'Age': age, 'Salary': salary}
data = pd.DataFrame(dic)
display(data)

pd.get_dummies(data, columns=['Country'], dtype=int)


,Country,Age,Salary
0,Taiwan,25,20000
1,Australia,30,32000
2,Ireland,45,59000
3,Australia,35,60000
4,Ireland,22,43000
5,Taiwan,36,52000


,Age,Salary,Country_Australia,Country_Ireland,Country_Taiwan
0,25,20000,0,0,1
1,30,32000,1,0,0
2,45,59000,0,1,0
3,35,60000,1,0,0
4,22,43000,0,1,0
5,36,52000,0,0,1
